# 프로젝트 2 - Weekend 1: RAG 평가 프레임워크와 하이브리드 검색

**프로젝트**: 검색형 RAG 기반 금융 상품(ETF) 추천 시스템

**학습 목표**:
1. ETF 데이터셋을 구축하고 벡터 스토어에 색인
2. Hit Rate, MRR, NDCG 등 검색 평가 지표 구현
3. BM25 + 벡터 하이브리드 검색으로 성능 개선
4. Query Expansion과 Multi-Query로 검색 품질 극대화

---
**김민아** | 260404

In [ ]:
# 환경 설정 및 라이브러리 설치
!pip install -q openai langchain langchain-openai langchain-community faiss-cpu \
    rank_bm25 pandas numpy matplotlib gradio python-dotenv tiktoken

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

client = OpenAI()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("\u2705 환경 설정 완료")

---
## 데이터 준비

아래 셀을 실행하면 ETF 금융 상품 데이터가 로드됩니다.

In [ ]:
# ============================================================
# ETF 금융 상품 샘플 데이터 (실습용)
# ============================================================
SAMPLE_ETF_DATA = [
    {"ticker": "KODEX200", "name": "KODEX 200", "category": "국내주식",
     "description": "KOSPI 200 지수를 추종하는 국내 대표 ETF. 삼성전자, SK하이닉스 등 대형주 중심으로 구성되어 있으며, 국내 주식시장 전체의 흐름을 반영합니다.",
     "expense_ratio": 0.15, "aum_billion": 58000, "risk_level": "중간",
     "returns": {"1m": 2.1, "3m": 5.4, "1y": 12.3, "3y": 28.5},
     "keywords": ["코스피", "대형주", "인덱스", "패시브"]},
    {"ticker": "TIGER미국S&P500", "name": "TIGER 미국 S&P500", "category": "해외주식",
     "description": "미국 S&P500 지수를 추종. 애플, 마이크로소프트, 아마존 등 미국 대형 기술주 포함. 환헤지 미적용으로 원/달러 환율 변동에 노출됩니다.",
     "expense_ratio": 0.07, "aum_billion": 42000, "risk_level": "중간",
     "returns": {"1m": 3.2, "3m": 8.1, "1y": 18.7, "3y": 45.2},
     "keywords": ["미국", "S&P500", "대형주", "기술주"]},
    {"ticker": "KODEX미국나스닥100", "name": "KODEX 미국나스닥100", "category": "해외주식",
     "description": "나스닥100 지수 추종. 기술 성장주 중심으로 애플, 엔비디아, 메타 등 포함. 고성장/고변동성 특성으로 공격적 투자자에게 적합합니다.",
     "expense_ratio": 0.09, "aum_billion": 35000, "risk_level": "높음",
     "returns": {"1m": 4.5, "3m": 12.3, "1y": 25.1, "3y": 62.8},
     "keywords": ["나스닥", "기술주", "성장주", "AI"]},
    {"ticker": "KODEX국고채10년", "name": "KODEX 국고채 10년", "category": "채권",
     "description": "대한민국 10년 만기 국고채를 추종하는 채권 ETF. 금리 인하 시 가격 상승, 안전자산 선호 시 수요 증가. 변동성이 낮아 안정적 투자에 적합합니다.",
     "expense_ratio": 0.05, "aum_billion": 12000, "risk_level": "낮음",
     "returns": {"1m": 0.8, "3m": 1.5, "1y": 4.2, "3y": 8.1},
     "keywords": ["국채", "채권", "안전자산", "금리"]},
    {"ticker": "TIGER금은혼합", "name": "TIGER 금은혼합", "category": "원자재",
     "description": "금과 은에 분산 투자하는 원자재 ETF. 인플레이션 헤지 수단으로 활용되며, 지정학적 리스크 시 안전자산으로 수요 증가합니다.",
     "expense_ratio": 0.39, "aum_billion": 3500, "risk_level": "중간",
     "returns": {"1m": 1.2, "3m": 3.8, "1y": 15.6, "3y": 35.2},
     "keywords": ["금", "은", "원자재", "인플레이션"]},
    {"ticker": "KODEX리츠", "name": "KODEX 한국부동산리츠인프라", "category": "부동산",
     "description": "국내 상장 리츠 및 인프라 펀드에 투자. 임대료 수입 기반의 안정적 배당 수익을 제공하며, 부동산 간접투자 수단으로 활용됩니다.",
     "expense_ratio": 0.09, "aum_billion": 4800, "risk_level": "중간",
     "returns": {"1m": -0.5, "3m": 2.1, "1y": 7.8, "3y": 15.3},
     "keywords": ["리츠", "부동산", "배당", "임대"]},
    {"ticker": "KODEX2차전지", "name": "KODEX 2차전지산업", "category": "테마",
     "description": "2차전지(배터리) 관련 기업에 집중 투자. LG에너지솔루션, 삼성SDI, 포스코퓨처엠 등 포함. 전기차 시장 성장에 따른 수혜가 기대됩니다.",
     "expense_ratio": 0.45, "aum_billion": 18000, "risk_level": "높음",
     "returns": {"1m": -2.3, "3m": -5.1, "1y": -12.4, "3y": 8.7},
     "keywords": ["2차전지", "배터리", "전기차", "테마"]},
    {"ticker": "TIGERBBD", "name": "TIGER 미국배당다우존스", "category": "배당",
     "description": "미국 고배당 우량주에 투자하는 ETF. 안정적인 배당 수익과 자본 이득을 동시에 추구합니다. 월배당 지급으로 현금흐름 관리에 유리합니다.",
     "expense_ratio": 0.01, "aum_billion": 52000, "risk_level": "낮음",
     "returns": {"1m": 1.8, "3m": 4.2, "1y": 10.5, "3y": 32.1},
     "keywords": ["배당", "미국", "월배당", "인컴"]},
    {"ticker": "KODEXKSM", "name": "KODEX 코스닥150", "category": "국내주식",
     "description": "코스닥 150 지수를 추종. 중소형 성장주 중심으로 바이오, IT, 게임 등 혁신 기업 포함. 코스피 대비 높은 변동성과 성장 잠재력을 가집니다.",
     "expense_ratio": 0.20, "aum_billion": 8500, "risk_level": "높음",
     "returns": {"1m": -1.2, "3m": 3.5, "1y": 8.9, "3y": 18.7},
     "keywords": ["코스닥", "중소형", "성장주", "바이오"]},
    {"ticker": "KOSEF단기자금", "name": "KOSEF 단기자금", "category": "머니마켓",
     "description": "초단기 채권 및 예금에 투자하는 MMF형 ETF. 원금 손실 위험이 극히 낮으며, 여유 자금 파킹 용도로 활용됩니다. 하루 단위 이자 발생.",
     "expense_ratio": 0.03, "aum_billion": 25000, "risk_level": "매우낮음",
     "returns": {"1m": 0.3, "3m": 0.9, "1y": 3.5, "3y": 10.2},
     "keywords": ["단기", "파킹", "안전", "예금"]},
]

# 사용자 프로필 데이터
SAMPLE_USER_PROFILES = [
    {"user_id": "U001", "name": "김초보", "level": "초보",
     "risk_tolerance": "낮음", "investment_goal": "안정적 수익",
     "monthly_budget": 500000, "preferred_categories": ["채권", "머니마켓"],
     "sample_queries": ["안전한 투자 상품 추천해주세요", "원금 손실 없는 ETF가 뭐가 있나요?"]},
    {"user_id": "U002", "name": "이중급", "level": "중급",
     "risk_tolerance": "중간", "investment_goal": "자산 증식",
     "monthly_budget": 2000000, "preferred_categories": ["국내주식", "해외주식"],
     "sample_queries": ["S&P500 추종 ETF 비교해주세요", "배당과 성장 균형 잡힌 포트폴리오 추천"]},
    {"user_id": "U003", "name": "박전문", "level": "전문",
     "risk_tolerance": "높음", "investment_goal": "공격적 수익",
     "monthly_budget": 10000000, "preferred_categories": ["테마", "해외주식"],
     "sample_queries": ["AI 관련 ETF 섹터 분석해줘", "나스닥100 vs 코스닥150 변동성 비교"]},
]

# 평가용 질의 데이터
SAMPLE_EVAL_QUERIES = [
    {"query": "초보자인데 안전한 투자 추천해주세요", "expected_tickers": ["KODEX국고채10년", "KOSEF단기자금", "TIGERBBD"]},
    {"query": "미국 기술주에 투자하고 싶어요", "expected_tickers": ["TIGER미국S&P500", "KODEX미국나스닥100"]},
    {"query": "월배당 받을 수 있는 ETF 있나요?", "expected_tickers": ["TIGERBBD", "KODEX리츠"]},
    {"query": "전기차 관련 투자 상품 알려주세요", "expected_tickers": ["KODEX2차전지"]},
    {"query": "인플레이션 헤지용 상품 추천", "expected_tickers": ["TIGER금은혼합", "KODEX리츠"]},
    {"query": "분산투자 포트폴리오 짜주세요", "expected_tickers": ["KODEX200", "TIGER미국S&P500", "KODEX국고채10년"]},
]

print(f"ETF 데이터 로드 완료: {len(SAMPLE_ETF_DATA)}개 상품, {len(SAMPLE_USER_PROFILES)}개 프로필, {len(SAMPLE_EVAL_QUERIES)}개 평가 질의")
for cat in sorted(set(e["category"] for e in SAMPLE_ETF_DATA)):
    cnt = sum(1 for e in SAMPLE_ETF_DATA if e["category"] == cat)
    print(f"  - {cat}: {cnt}개")

---
## 문제 1: ETF 사용자 페르소나 정의

ETF 추천 시스템의 사용자 페르소나 3가지(초보/중급/전문)를 정의하세요.

**요구사항:**
- `personas` 딕셔너리: 키='초보'/'중급'/'전문'
- 각 페르소나에 `description`(설명)과 `queries`(질의 리스트) 포함
- 각 질의에 `query`(질문 텍스트)와 `expected_category`(기대 카테고리) 포함
- `project2_data/query_set.json`으로 저장

각 투자 레벨에 맞는 질의를 만들었다. 초보 투자자는 안정적인 상품을, 전문가는 섹터별 비교 같은 질의를 하도록 설정.

In [ ]:
import os
os.makedirs("project2_data", exist_ok=True)

# 문제 1: 사용자 페르소나 정의 - 투자 레벨별로 다른 질의 패턴을 반영
personas = {
    "초보": {
        "description": "투자 경험이 거의 없고, 원금 보존을 최우선시하는 투자자",
        "queries": [
            {"query": "안전한 투자 상품 추천해주세요", "expected_category": ["채권", "머니마켓"]},
            {"query": "원금 손실 위험 없는 ETF", "expected_category": ["머니마켓"]},
            {"query": "적금보다 나은 안전한 투자", "expected_category": ["채권", "배당"]},
        ]
    },
    "중급": {
        "description": "기본적인 투자 지식이 있고, 적절한 위험을 감수하며 자산 증식을 추구하는 투자자",
        "queries": [
            {"query": "S&P500 추종 ETF 비교", "expected_category": ["해외주식"]},
            {"query": "배당과 성장 균형 잡힌 포트폴리오", "expected_category": ["배당", "국내주식"]},
            {"query": "분산투자 가능한 ETF 조합", "expected_category": ["국내주식", "해외주식", "채권"]},
        ]
    },
    "전문": {
        "description": "투자 경험이 풍부하고, 높은 수익을 위해 높은 변동성을 감수할 수 있는 투자자",
        "queries": [
            {"query": "AI 반도체 관련 ETF 섹터 분석", "expected_category": ["테마", "해외주식"]},
            {"query": "레버리지 ETF 단기 트레이딩 전략", "expected_category": ["레버리지"]},
            {"query": "나스닥100 vs 코스닥150 변동성 비교", "expected_category": ["해외주식", "국내주식"]},
        ]
    },
}

# JSON 파일로 저장
with open("project2_data/query_set.json", "w") as f:
    json.dump(personas, f, ensure_ascii=False, indent=2)

print(f"\u2705 {sum(len(p['queries']) for p in personas.values())}개 질의 저장 완료")

---
## 문제 2: 추가 ETF 문서 생성

5개 추가 ETF(ESG, 2차전지, 헬스케어, 리츠, 원자재)를 정의하고 LLM으로 설명을 생성하세요.

**요구사항:**
- `additional_etfs` 리스트: 5개 ETF (name, category, market)
- `risk_map` 딕셔너리: 카테고리별 위험도 매핑
- 각 ETF에 대해 `client.chat.completions.create()`로 설명 생성
- `etf_documents`에 추가 후 카테고리별 통계 출력

LLM한테 각 ETF 설명을 생성하게 해서 문서를 만드는 부분. risk_map으로 카테고리별 위험도도 같이 관리한다.

In [ ]:
# 기존 ETF 데이터를 etf_documents 리스트로 변환
etf_documents = []
for etf in SAMPLE_ETF_DATA:
    etf_documents.append({
        "name": etf["name"],
        "category": etf["category"],
        "market": "국내",
        "content": etf["description"]
    })

# 문제 2: 추가 ETF 5개 정의
additional_etfs = [
    {"name": "TIGER ESG리더스", "category": "ESG", "market": "국내"},
    {"name": "KODEX 2차전지산업", "category": "2차전지", "market": "국내"},
    {"name": "TIGER 헬스케어", "category": "헬스케어", "market": "국내"},
    {"name": "KODEX 한국부동산리츠인프라", "category": "리츠", "market": "국내"},
    {"name": "KODEX 골드선물", "category": "원자재", "market": "글로벌"},
]

# 카테고리별 위험도(1~5) - 1이 가장 안전
risk_map = {
    "ESG": 3, "2차전지": 4, "헬스케어": 3, "리츠": 2, "원자재": 3,
    "인덱스": 2, "섹터": 4, "배당": 2, "레버리지": 5, "테마": 4, "채권": 1, "자산배분": 2
}

# 각 ETF에 대해 LLM으로 설명 문서 생성
for etf in additional_etfs:
    prompt = f"""다음 ETF에 대한 상세 설명을 작성해주세요:
    - ETF명: {etf['name']}
    - 카테고리: {etf['category']}
    - 시장: {etf['market']}

    다음 항목을 포함해주세요:
    1. 투자 전략 (3-4문장)
    2. 주요 편입 종목 (5개)
    3. 수수료 및 비용 (총보수)
    4. 적합한 투자자 유형
    5. 주의사항

    한국어로 300-400자 내외로 작성해주세요."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )

    doc_text = response.choices[0].message.content
    etf_documents.append({
        "name": etf["name"],
        "category": etf["category"],
        "market": etf["market"],
        "content": doc_text
    })
    print(f"\u2705 {etf['name']} 문서 생성 완료 (위험도: {risk_map.get(etf['category'], '?')})")

df = pd.DataFrame(etf_documents)
print(f"\n카테고리별 수:\n{df['category'].value_counts()}")

---
## 벡터 스토어 구축

생성된 ETF 문서들을 FAISS 벡터 스토어에 색인합니다.

In [ ]:
# Document 객체로 변환하고 FAISS 색인
docs = []
for i, etf in enumerate(etf_documents):
    doc = Document(
        page_content=etf["content"],
        metadata={"doc_id": i, "name": etf["name"], "category": etf["category"], "market": etf.get("market", "국내")}
    )
    docs.append(doc)

# FAISS 벡터스토어 생성
vectorstore = FAISS.from_documents(docs, embeddings)
vs = vectorstore  # 편의를 위한 별칭

print(f"\u2705 FAISS 벡터 스토어 구축 완료: {len(docs)}개 문서 색인")

---
## 평가 데이터셋 및 기본 평가 함수 준비

평가용 질의-정답 쌍과 기본 평가 함수(Hit Rate, MRR, NDCG)를 미리 정의합니다.

In [ ]:
# 평가 데이터 변환: SAMPLE_EVAL_QUERIES의 ticker를 doc_id로 매핑
# (원본 데이터에서 ticker -> 이름 기준으로 doc_id를 찾아야 함)
ticker_to_docid = {}
for i, etf in enumerate(SAMPLE_ETF_DATA):
    ticker_to_docid[etf["ticker"]] = i

eval_data = []
for item in SAMPLE_EVAL_QUERIES:
    doc_ids = [ticker_to_docid[t] for t in item["expected_tickers"] if t in ticker_to_docid]
    eval_data.append({
        "query": item["query"],
        "relevant_doc_ids": doc_ids
    })

print(f"\u2705 평가 데이터 준비 완료: {len(eval_data)}개 질의")
for item in eval_data:
    print(f"  Q: {item['query']} -> doc_ids: {item['relevant_doc_ids']}")

In [ ]:
# 기본 평가 함수: Hit Rate@K, MRR@K, NDCG@K

def hit_rate_at_k(vectorstore, eval_data, k=5):
    """상위 K개 결과에 정답이 하나라도 포함되면 hit"""
    hits = 0
    for item in eval_data:
        results = vectorstore.similarity_search(item["query"], k=k)
        retrieved_ids = [r.metadata["doc_id"] for r in results]
        if any(rid in item["relevant_doc_ids"] for rid in retrieved_ids):
            hits += 1
    return hits / len(eval_data)

def mrr_at_k(vectorstore, eval_data, k=5):
    """첫 번째 정답의 역수 순위 평균"""
    rr_sum = 0
    for item in eval_data:
        results = vectorstore.similarity_search(item["query"], k=k)
        retrieved_ids = [r.metadata["doc_id"] for r in results]
        for rank, rid in enumerate(retrieved_ids, 1):
            if rid in item["relevant_doc_ids"]:
                rr_sum += 1.0 / rank
                break
    return rr_sum / len(eval_data)

def ndcg_at_k(relevances, k):
    """단일 질의에 대한 NDCG 계산"""
    # DCG 계산
    dcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(relevances[:k]))
    # IDCG 계산 (이상적 순서)
    ideal = sorted(relevances, reverse=True)[:k]
    idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0

print("\u2705 평가 함수 정의 완료 (hit_rate_at_k, mrr_at_k, ndcg_at_k)")

---
## 문제 3: 멀티-관련성 질의 생성

하나의 질의에 여러 ETF가 관련되는 멀티-관련성 질의를 생성하세요.

**요구사항:**
- `multi_queries` 리스트: 각 항목에 `query`, `relevant`(관련 문서 리스트), `irrelevant` 포함
- `cats` 변수에 카테고리 분포 저장
- matplotlib로 카테고리 분포 바 차트 시각화
- CSV 파일로 저장

멀티-관련성 질의는 하나의 질문에 여러 ETF가 관련되는 경우를 평가하기 위한 것. score는 관련도(graded relevance)를 나타냄.

In [ ]:
import os
os.makedirs("project2_data/evaluation", exist_ok=True)
os.makedirs("project2_data/raw", exist_ok=True)

# LLM으로 평가용 질의-정답 쌍 생성
eval_dataset = []

for i, doc in enumerate(etf_documents[:10]):  # 처음 10개 ETF에 대해
    prompt = f"""다음 ETF 문서를 읽고, 이 ETF를 찾기 위한 자연스러운 질의 3개를 만들어주세요.

    ETF: {doc['name']}
    카테고리: {doc['category']}
    내용: {doc['content'][:300]}

    JSON 배열로 반환: ["질의1", "질의2", "질의3"]"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        response_format={"type": "json_object"}
    )

    try:
        result = json.loads(response.choices[0].message.content)
        # LLM 응답의 키가 다양할 수 있으므로 유연하게 처리
        queries = result.get("queries", result.get("질의", list(result.values())[0]))
        if isinstance(queries, list):
            for q in queries:
                eval_dataset.append({
                    "query": q,
                    "relevant_doc_ids": [i],
                    "relevant_doc_names": [doc["name"]],
                    "category": doc["category"]
                })
    except:
        pass
    print(f"\u2705 {doc['name']}: 질의 생성 완료")

# 저장
with open("project2_data/evaluation/eval_queries.json", "w") as f:
    json.dump(eval_dataset, f, ensure_ascii=False, indent=2)

print(f"\n총 {len(eval_dataset)}개 평가 질의 생성")
for item in eval_dataset[:3]:
    print(f"  Q: {item['query']}")
    print(f"  A: {item['relevant_doc_names']}\n")

In [ ]:
import matplotlib.pyplot as plt

# 멀티-관련성 질의: 하나의 질문에 여러 문서가 관련됨
# score는 관련도 점수 (3: 매우 관련, 2: 관련, 1: 약간 관련)
multi_queries = [
    {
        "query": "분산투자에 좋은 안전한 포트폴리오 구성 추천",
        "relevant": [{"doc_id": 0, "score": 2}, {"doc_id": 8, "score": 3}, {"doc_id": 9, "score": 2}],
        "irrelevant": [6]
    },
    {
        "query": "미국 시장에 투자할 수 있는 ETF 비교",
        "relevant": [{"doc_id": 1, "score": 3}, {"doc_id": 2, "score": 3}, {"doc_id": 5, "score": 2}],
        "irrelevant": [3, 8]
    },
    {
        "query": "배당 수익과 안정성을 동시에 추구하는 ETF",
        "relevant": [{"doc_id": 4, "score": 3}, {"doc_id": 5, "score": 2}, {"doc_id": 8, "score": 2}],
        "irrelevant": [2, 6]
    },
]

# 카테고리 분포 시각화
cats = [item['category'] for item in eval_dataset]
cat_counts = pd.Series(cats).value_counts()

plt.figure(figsize=(8, 4))
cat_counts.plot(kind='bar')
plt.title('평가 질의 카테고리 분포')
plt.xlabel('카테고리')
plt.ylabel('질의 수')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# CSV로 저장
pd.DataFrame(eval_dataset).to_csv("project2_data/evaluation/eval_queries.csv", index=False)
print("\u2705 CSV 저장 완료")

---
## 문제 4: MMR 검색 구현

벡터 스토어에서 MMR(Maximal Marginal Relevance) 검색을 구현하고 유사도 검색과 비교하세요.

**요구사항:**
- `format_results(results, method_name)` 함수: 검색 결과를 포맷팅하여 출력
- Similarity Search와 MMR Search 각각 실행 후 시간 측정
- k값(3, 5, 10)별 카테고리 다양성 비교

MMR은 관련성과 다양성을 동시에 고려하는 검색 방법이다. 유사도 검색은 비슷한 결과가 몰릴 수 있는데, MMR은 다양한 카테고리의 결과를 가져오는 장점이 있다.

In [ ]:
import time

def format_results(results, method_name):
    """검색 결과를 보기 좋게 출력하는 헬퍼 함수"""
    print(f"\n{method_name} 결과:")
    for i, item in enumerate(results, 1):
        # similarity_search_with_score는 (doc, score) 튜플 반환
        if isinstance(item, tuple) and len(item) == 2:
            doc, score = item
            print(f"  {i}. [{score:.4f}] {doc.metadata['name']} ({doc.metadata['category']})")
        else:
            # MMR은 Document 객체만 반환
            print(f"  {i}. {item.metadata['name']} ({item.metadata['category']})")

query = "분산 투자에 적합한 ETF"

# 1) Similarity Search
start = time.time()
sim_results = vectorstore.similarity_search_with_score(query, k=5)
sim_time = time.time() - start
format_results(sim_results, f"Similarity Search ({sim_time:.3f}s)")

# 2) MMR Search - 다양성 고려
start = time.time()
mmr_results = vectorstore.max_marginal_relevance_search(query, k=5, fetch_k=10)
mmr_time = time.time() - start
format_results([(doc, 0) for doc in mmr_results], f"MMR Search ({mmr_time:.3f}s)")

# k값별 카테고리 다양성 비교
print("\nk값별 카테고리 다양성 비교:")
for k in [3, 5, 10]:
    sim = vectorstore.similarity_search(query, k=k)
    mmr = vectorstore.max_marginal_relevance_search(query, k=k, fetch_k=max(k*2, 10))
    sim_cats = set(d.metadata['category'] for d in sim)
    mmr_cats = set(d.metadata['category'] for d in mmr)
    print(f"  k={k}: Similarity {len(sim_cats)}개 카테고리 vs MMR {len(mmr_cats)}개 카테고리")

---
## 문제 5: Precision@K와 Recall@K 구현

Precision@K와 Recall@K 평가 지표를 구현하세요.

**요구사항:**
- `precision_at_k(vectorstore, eval_data, k=5)` 함수 구현
- `recall_at_k(vectorstore, eval_data, k=5)` 함수 구현
- k=1~10에서 Hit Rate, MRR, Precision, Recall 4개 지표를 그래프로 비교

- Precision@K = (검색된 것 중 정답 수) / K  -> 검색 결과의 정확도
- Recall@K = (검색된 것 중 정답 수) / (전체 정답 수) -> 정답을 얼마나 잘 찾았는지

보통 K가 커지면 Recall은 올라가고 Precision은 내려가는 trade-off 관계가 있다.

In [ ]:
import matplotlib.pyplot as plt

def precision_at_k(vectorstore, eval_data, k=5):
    """검색된 K개 중 정답의 비율"""
    precisions = []
    for item in eval_data:
        results = vectorstore.similarity_search(item["query"], k=k)
        retrieved_ids = [r.metadata["doc_id"] for r in results]
        relevant_retrieved = sum(1 for rid in retrieved_ids if rid in item["relevant_doc_ids"])
        precisions.append(relevant_retrieved / k)
    return np.mean(precisions)

def recall_at_k(vectorstore, eval_data, k=5):
    """전체 정답 중 검색된 비율"""
    recalls = []
    for item in eval_data:
        results = vectorstore.similarity_search(item["query"], k=k)
        retrieved_ids = [r.metadata["doc_id"] for r in results]
        relevant_retrieved = sum(1 for rid in retrieved_ids if rid in item["relevant_doc_ids"])
        total_relevant = len(item["relevant_doc_ids"])
        recalls.append(relevant_retrieved / total_relevant if total_relevant > 0 else 0)
    return np.mean(recalls)

# K=1~10 구간에서 4가지 지표 비교 그래프
ks = range(1, 11)
hrs = [hit_rate_at_k(vectorstore, eval_data, k) for k in ks]
mrrs = [mrr_at_k(vectorstore, eval_data, k) for k in ks]
precs = [precision_at_k(vectorstore, eval_data, k) for k in ks]
recs = [recall_at_k(vectorstore, eval_data, k) for k in ks]

plt.figure(figsize=(10, 6))
plt.plot(ks, hrs, 'o-', label='Hit Rate')
plt.plot(ks, mrrs, 's-', label='MRR')
plt.plot(ks, precs, '^-', label='Precision')
plt.plot(ks, recs, 'D-', label='Recall')
plt.xlabel('K')
plt.ylabel('Score')
plt.title('검색 평가 지표 비교 (K=1~10)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 문제 6: NDCG 평가 리포트

Graded relevance(0, 1, 2, 3)를 사용한 NDCG 평가 리포트를 작성하세요.

**요구사항:**
- `report` 딕셔너리에 k=1,3,5,10별 Hit Rate, MRR, NDCG 저장
- JSON 파일로 저장 (`baseline_report.json`)
- 결과를 표 형태로 출력

NDCG는 순서까지 고려하는 지표. 정답이 상위에 올수록 점수가 높아진다. Hit Rate/MRR과 함께 비교하면 검색 성능을 더 잘 파악할 수 있음.

In [ ]:
# 문제 6: 베이스라인 평가 리포트 - k별로 3가지 지표를 한번에 정리
report = {'baseline': {}}

for k in [1, 3, 5, 10]:
    hr = hit_rate_at_k(vectorstore, eval_data, k)
    mrr = mrr_at_k(vectorstore, eval_data, k)

    # NDCG 계산: 각 질의별로 relevance 리스트 생성 후 평균
    ndcg_scores = []
    for item in eval_data:
        results = vs.similarity_search(item["query"], k=k)
        # 관련 문서면 1, 아니면 0 (binary relevance)
        rels = [1 if r.metadata["doc_id"] in item["relevant_doc_ids"] else 0 for r in results]
        ndcg_scores.append(ndcg_at_k(rels, k))

    report['baseline'][f'k={k}'] = {
        'hit_rate': round(hr, 4),
        'mrr': round(mrr, 4),
        'ndcg': round(np.mean(ndcg_scores), 4)
    }

# JSON으로 저장
with open("project2_data/evaluation/baseline_report.json", "w") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

# 테이블 형식 출력
print(f"{'K':<6} | {'Hit Rate':>10} | {'MRR':>10} | {'NDCG':>10}")
print("-" * 45)
for k_str, metrics in report['baseline'].items():
    print(f"{k_str:<6} | {metrics['hit_rate']:>10.4f} | {metrics['mrr']:>10.4f} | {metrics['ndcg']:>10.4f}")

print(f"\n\u2705 baseline_report.json 저장 완료")

---
## BM25 검색 구축

키워드 기반 BM25 검색을 구축합니다. 문제 7~9에서 활용됩니다.

In [ ]:
from rank_bm25 import BM25Okapi

# BM25 인덱스 구축 - 문서를 토큰화해서 넣어야 함
corpus_tokens = [doc["content"].split() for doc in etf_documents]
bm25 = BM25Okapi(corpus_tokens)

def bm25_search(query, k=5):
    """BM25 키워드 검색"""
    query_tokens = query.split()
    scores = bm25.get_scores(query_tokens)
    top_indices = np.argsort(scores)[::-1][:k]
    results = []
    for idx in top_indices:
        results.append({
            "doc_id": idx,
            "name": etf_documents[idx]["name"],
            "score": float(scores[idx])
        })
    return results

# 하이브리드 검색 함수 (FAISS + BM25 결합)
def hybrid_search(query, alpha=0.5, k=5):
    """alpha: FAISS 가중치 (1-alpha: BM25 가중치)"""
    # FAISS 결과
    faiss_results = vs.similarity_search_with_score(query, k=k)
    faiss_scores = {}
    for doc, score in faiss_results:
        doc_id = doc.metadata["doc_id"]
        # FAISS의 L2 거리를 유사도로 변환 (거리가 작을수록 유사)
        faiss_scores[doc_id] = 1.0 / (1.0 + score)

    # BM25 결과
    bm25_results = bm25_search(query, k=k)
    bm25_scores = {}
    max_bm25 = max(r["score"] for r in bm25_results) if bm25_results else 1
    for r in bm25_results:
        bm25_scores[r["doc_id"]] = r["score"] / max_bm25 if max_bm25 > 0 else 0

    # 점수 결합
    all_ids = set(faiss_scores.keys()) | set(bm25_scores.keys())
    combined = []
    for doc_id in all_ids:
        f_score = faiss_scores.get(doc_id, 0)
        b_score = bm25_scores.get(doc_id, 0)
        final_score = alpha * f_score + (1 - alpha) * b_score
        combined.append((doc_id, final_score, etf_documents[doc_id]["name"]))

    combined.sort(key=lambda x: x[1], reverse=True)
    return combined[:k]

print("\u2705 BM25 + 하이브리드 검색 함수 준비 완료")

# 간단 테스트
test_q = "배당 수익이 높은 안전한 ETF"
print(f"\nBM25 검색: '{test_q}'")
for r in bm25_search(test_q, k=3):
    print(f"  {r['name']} (score: {r['score']:.4f})")

---
## 문제 7: BM25 vs FAISS 비교 평가

BM25 검색 결과를 FAISS 검색 결과와 비교 평가하세요.

**요구사항:**
- `bm25_hit_rate(eval_data, k=5)` 함수 구현
- `bm25_mrr(eval_data, k=5)` 함수 구현
- BM25와 FAISS의 Hit Rate, MRR 비교 테이블 출력

BM25는 키워드 매칭 기반, FAISS는 의미적 유사도 기반. 어떤 질의에서 어떤 방법이 더 나은지 비교해본다.

In [ ]:
# 문제 7: BM25 평가 함수 구현
def bm25_hit_rate(eval_data, k=5):
    """BM25 검색의 Hit Rate"""
    hits = 0
    for item in eval_data:
        results = bm25_search(item["query"], k)
        retrieved_ids = [r["doc_id"] for r in results]
        if any(rid in item["relevant_doc_ids"] for rid in retrieved_ids):
            hits += 1
    return hits / len(eval_data)

def bm25_mrr(eval_data, k=5):
    """BM25 검색의 MRR"""
    rr_sum = 0
    for item in eval_data:
        results = bm25_search(item["query"], k)
        retrieved_ids = [r["doc_id"] for r in results]
        for rank, rid in enumerate(retrieved_ids, 1):
            if rid in item["relevant_doc_ids"]:
                rr_sum += 1.0 / rank
                break
    return rr_sum / len(eval_data)

# 비교 테이블 출력
print(f"{'방법':<12} | {'K':>3} | {'Hit Rate':>10} | {'MRR':>10}")
print("-" * 45)
for k in [1, 3, 5, 10]:
    faiss_hr = hit_rate_at_k(vectorstore, eval_data, k)
    faiss_mrr = mrr_at_k(vectorstore, eval_data, k)
    bm25_hr = bm25_hit_rate(eval_data, k)
    bm25_m = bm25_mrr(eval_data, k)
    print(f"{'FAISS':<12} | {k:>3} | {faiss_hr:>10.4f} | {faiss_mrr:>10.4f}")
    print(f"{'BM25':<12} | {k:>3} | {bm25_hr:>10.4f} | {bm25_m:>10.4f}")
    print("-" * 45)

---
## 문제 8: Alpha 최적화

Alpha 값을 0.0~1.0까지 0.1 단위로 변경하며 Hit Rate를 측정하고 최적값을 찾으세요.

**요구사항:**
- alpha를 0.0~1.0 범위에서 0.1 간격으로 탐색
- 각 alpha별 Hit Rate@5 출력
- 최적 alpha와 Hit Rate를 `checkpoint` 딕셔너리에 저장
- `project2_data/checkpoints/hybrid_config.json`으로 저장

alpha=0이면 BM25만, alpha=1이면 FAISS만 사용. 둘을 적절히 섞는 최적 비율을 찾는게 목표.

In [ ]:
os.makedirs("project2_data/checkpoints", exist_ok=True)

# 문제 8: alpha 값별 Hit Rate 측정하여 최적값 탐색
best_alpha, best_hr = 0, 0

print(f"{'Alpha':>6} | {'Hit Rate@5':>12}")
print("-" * 25)

for a in np.arange(0, 1.1, 0.1):
    a = round(a, 1)  # 부동소수점 오차 방지
    hits = 0
    for item in eval_data:
        results = hybrid_search(item['query'], alpha=a, k=5)
        retrieved_ids = [r[0] for r in results]
        if any(rid in item['relevant_doc_ids'] for rid in retrieved_ids):
            hits += 1
    hr = hits / len(eval_data)
    print(f"{a:>6.1f} | {hr:>12.4f}")
    if hr > best_hr:
        best_alpha, best_hr = a, hr

# 최적값 저장
checkpoint = {'best_alpha': best_alpha, 'best_hr': round(best_hr, 4)}
with open("project2_data/checkpoints/hybrid_config.json", "w") as f:
    json.dump(checkpoint, f, ensure_ascii=False, indent=2)

print(f"\n최적 alpha: {best_alpha} (Hit Rate: {best_hr:.4f})")
print("\u2705 hybrid_config.json 저장 완료")

---
## 문제 9: 도메인 특화 동의어 사전

도메인 특화 동의어 사전을 만들고 쿼리 확장 성능을 비교하세요.

**요구사항:**
- `finance_synonyms` 딕셔너리: ETF, 배당, 안정, 성장, 미국 등의 동의어
- `synonym_expand(query)` 함수: 동의어로 확장된 쿼리 리스트 반환
- Baseline vs Synonym 확장 Hit Rate 비교

사용자가 "안전한 투자"라고 검색해도 "보수적", "저위험" 같은 동의어로 확장하면 더 많은 관련 문서를 찾을 수 있다.

In [ ]:
# 문제 9: 금융 도메인 동의어 사전
finance_synonyms = {
    'ETF': ['상장지수펀드', '인덱스펀드', '지수추종'],
    '배당': ['분배금', '배당금', '인컴', '이자수익'],
    '안정': ['안전', '보수적', '저위험', '원금보존'],
    '성장': ['그로스', '공격적', '고수익', '고성장'],
    '미국': ['해외', '글로벌', '나스닥', 'S&P'],
}

def synonym_expand(query):
    """동의어 사전 기반 쿼리 확장: 원본 + 동의어 치환 쿼리들 반환"""
    expanded = [query]  # 원본 쿼리 포함
    for keyword, synonyms in finance_synonyms.items():
        if keyword in query:
            for syn in synonyms:
                expanded.append(query.replace(keyword, syn))
    return expanded

# 동의어 확장 예시
test_query = "안정적인 ETF 추천"
print(f"원본: {test_query}")
print(f"확장: {synonym_expand(test_query)}")

# Baseline vs Synonym Hit Rate 비교
baseline_hits, synonym_hits = 0, 0
for item in eval_data:
    # Baseline: 하이브리드 검색 그대로
    results = hybrid_search(item['query'], alpha=0.5, k=5)
    if any(r[0] in item['relevant_doc_ids'] for r in results):
        baseline_hits += 1

    # Synonym: 확장된 쿼리들 결과를 합쳐서 상위 5개
    expanded = synonym_expand(item['query'])
    all_results = {}
    for eq in expanded:
        for doc_id, score, name in hybrid_search(eq, alpha=0.5, k=5):
            # 같은 doc_id면 더 높은 점수로 갱신
            if doc_id not in all_results or score > all_results[doc_id][1]:
                all_results[doc_id] = (doc_id, score, name)
    top_results = sorted(all_results.values(), key=lambda x: x[1], reverse=True)[:5]
    if any(r[0] in item['relevant_doc_ids'] for r in top_results):
        synonym_hits += 1

print(f"\nBaseline Hit Rate: {baseline_hits/len(eval_data):.4f}")
print(f"Synonym  Hit Rate: {synonym_hits/len(eval_data):.4f}")

---
## 문제 10: 커스텀 Multi-Query Retriever

커스텀 프롬프트로 Multi-Query Retriever를 설정하세요.

**요구사항:**
- `custom_prompt` PromptTemplate 정의 (ETF 검색 전문가 역할)
- `retriever_custom` = MultiQueryRetriever 생성
- `retriever_custom.invoke()`로 검색 실행 후 결과 출력

Multi-Query는 하나의 질문을 여러 관점에서 재작성해서 검색하는 방법. 사용자의 의도를 다양하게 해석해서 더 넓은 범위의 관련 문서를 찾을 수 있다.

In [ ]:
from langchain.prompts import PromptTemplate
from langchain.retrievers.multi_query import MultiQueryRetriever

# ETF 검색에 특화된 프롬프트 - 다양한 관점에서 질의를 재작성
custom_prompt = PromptTemplate(
    input_variables=['question'],
    template="""당신은 ETF 금융 상품 검색 전문가입니다.
다음 질문을 서로 다른 관점에서 3가지로 재작성하세요.
각 질의는 ETF 검색에 최적화되어야 합니다.

원래 질문: {question}

재작성된 질의 (한 줄에 하나씩):"""
)

# Multi-Query Retriever 생성
retriever_custom = MultiQueryRetriever.from_llm(
    retriever=vs.as_retriever(search_kwargs={"k": 5}),
    llm=llm,
    prompt=custom_prompt
)

# 검색 실행
results_custom = retriever_custom.invoke("노후 대비 안정적 투자")
print(f"커스텀 Multi-Query 결과: {len(results_custom)}개")
for doc in results_custom:
    print(f"  - {doc.metadata['name']}: {doc.page_content[:80]}...")

---
## 문제 11: 검색 비교 대시보드

모든 검색 방법의 결과를 나란히 비교하는 Gradio UI를 만드세요.

**요구사항:**
- `full_comparison(query, top_k)` 함수: FAISS/BM25/Hybrid 결과 비교
- `show_history()` 함수: 최근 검색 이력 표시
- `gr.Blocks`로 탭 UI 구성 (검색 탭 + 이력 탭)

최종 결과물! 3가지 검색 방법(FAISS, BM25, Hybrid)을 한 화면에서 비교할 수 있는 대시보드.

In [ ]:
import gradio as gr

search_history = []

def full_comparison(query, top_k):
    """FAISS, BM25, Hybrid 3가지 검색 결과를 나란히 비교"""
    top_k = int(top_k)
    search_history.append(query)

    output = f"질의: {query}\n{'='*60}\n\n"

    # 1) FAISS 벡터 검색
    faiss_results = vs.similarity_search_with_score(query, k=top_k)
    output += "[FAISS 벡터 검색]\n"
    for i, (doc, score) in enumerate(faiss_results, 1):
        output += f"  {i}. [{score:.4f}] {doc.metadata['name']} ({doc.metadata['category']})\n"

    # 2) BM25 키워드 검색
    bm25_results = bm25_search(query, top_k)
    output += "\n[BM25 키워드 검색]\n"
    for i, r in enumerate(bm25_results, 1):
        output += f"  {i}. [{r['score']:.4f}] {r['name']}\n"

    # 3) 하이브리드 검색
    hybrid_results = hybrid_search(query, alpha=0.5, k=top_k)
    output += "\n[하이브리드 검색 (alpha=0.5)]\n"
    for i, (doc_id, score, name) in enumerate(hybrid_results, 1):
        output += f"  {i}. [{score:.4f}] {name}\n"

    return output

def show_history():
    """검색 이력 보여주기"""
    if not search_history:
        return "검색 이력이 없습니다."
    return "\n".join(f"{i+1}. {q}" for i, q in enumerate(search_history))

# Gradio 대시보드 - 탭 구성
with gr.Blocks(title="ETF 검색 비교 대시보드") as demo:
    gr.Markdown("# ETF 검색 비교 대시보드")

    with gr.Tab("검색"):
        query_input = gr.Textbox(label="검색 질의", placeholder="예: 배당 수익률 높은 안전한 ETF")
        top_k_slider = gr.Slider(1, 10, value=5, step=1, label="결과 수 (K)")
        search_btn = gr.Button("검색")
        output = gr.Textbox(label="비교 결과", lines=20)
        search_btn.click(full_comparison, [query_input, top_k_slider], output)

    with gr.Tab("검색 이력"):
        history_btn = gr.Button("이력 조회")
        history_output = gr.Textbox(label="최근 검색 이력", lines=10)
        history_btn.click(show_history, [], history_output)

demo.launch(share=True)

---

## Weekend 1 요약

### 달성한 것들
1. **ETF 데이터셋 구축**: 15개 ETF 문서 생성 및 벡터화
2. **평가 프레임워크**: Hit Rate, MRR, NDCG 구현
3. **BM25 검색**: 키워드 기반 검색 구축
4. **하이브리드 검색**: 벡터+BM25 앙상블
5. **고급 기법**: Query Expansion, Multi-Query
6. **Gradio 대시보드**: 인터랙티브 검색 UI

### Weekend 2 예고
- 리랭킹과 답변 품질 평가
- BLEU, ROUGE, BERTScore, LLM-as-Judge 평가
- 종합 평가 파이프라인과 품질 대시보드